### ID3 Decision Tree Implementation

In [16]:
import math
from collections import Counter
import pandas as pd
import random
from src.game import PopOut
import ast

In [5]:
def get_entropy(labels):
    if not labels:
        return 0.0
    else:
        entropy = 0.0
        counts = Counter(labels)
        total  = len(labels)
        for count in counts.values():
            if count > 0:
                entropy -= (count / total) * math.log2(count / total)
        return entropy
    
def information_gain(data, labels, attribute_idx):
    ''' Returns the information gain of a certain attirbute'''
    h_c = get_entropy(labels)
    # Group labels by attribute value
    subsets = {}
    for i, row in enumerate(data):
        key = row[attribute_idx]
        subsets.setdefault(key, []).append(labels[i])
    total = len(labels)
    conditional_entropy = 0.0
    for subset_labels in subsets.values():
        probability = len(subset_labels) / total
        conditional_entropy += probability * get_entropy(subset_labels)
    gain = h_c - conditional_entropy
    return gain

In [6]:
class DecisionTreeNode:
    def __init__(self, is_leaf=False, label=None, feature_idx=None):
        self.is_leaf      = is_leaf
        self.label        = label
        self.feature_idx  = feature_idx
        self.children     = {}

def id3(data, labels, feature_indices, depth=0, max_depth=None):
    # Base cases
    # 1 - All the examples in the node have the same label: it is a leaf
    if (len(set(labels))) == 1: 
        return DecisionTreeNode(is_leaf=True, label=labels[0])
    # 2 - No attributes for splitting 
    if not feature_indices:
        majority = Counter(labels).most_common(1)[0][0]
        return DecisionTreeNode(is_leaf=True, label=majority)
    
    # Choosing the attribute with the highest information gain
    best_idx = None
    best_gain = 0.0
    for idx in feature_indices:
        gain = information_gain(data, labels, idx)
        if gain > best_gain:
            best_gain = gain
            best_idx = idx
    
    # If no best index found, use majority label
    if best_idx is None:
        majority = Counter(labels).most_common(1)[0][0]
        return DecisionTreeNode(is_leaf=True, label=majority)
    
    node = DecisionTreeNode(feature_idx = best_idx)
    values = set(row[best_idx] for row in data)
    remaining = [idx for idx in feature_indices if idx != best_idx]
    
    # Construct subtree
    for val in values:
        sub_data   = [data[i] for i in range(len(data)) if data[i][best_idx] == val]
        sub_labels = [labels[i] for i in range(len(labels)) if data[i][best_idx] == val]
        # Recursive call
        node.children[val] = id3(sub_data, sub_labels, remaining, depth + 1, max_depth)
    return node

def predict(tree, value):
    node = tree
    while not node.is_leaf:
        if node.feature_idx is None:
            return None
        val = value[node.feature_idx]
        if val not in node.children:
            return None  # valor desconhecido
        node = node.children[val]
    return node.label

def accuracy(tree, data, labels):
    total = len(labels)
    correct = 0
    for i, ex in enumerate(data):
        if predict(tree, ex) == labels[i]:
            correct += 1
    if not labels:
        return 0.0
    else:
        return correct / total

Test the tree on the iris dataset

In [7]:
df_iris = pd.read_csv('iris.csv')
features = df_iris.columns[1:-1]
label = df_iris.columns[-1]

def learn_bins(data, n_bins = 3):
    '''Learn bin boundaries from data'''
    min_val = min(data)
    max_val = max(data)
    bin_size = (max_val - min_val) / n_bins
    return [min_val + bin_size, min_val + 2 * bin_size]

def discretize_value(value, bins):
    '''Discretize a single value using predefined bins'''
    if value <= bins[0]:
        return "low"
    elif value <= bins[1]:
        return "medium"
    else:
        return "high"

In [8]:
# Train and test splitting with separate discretization
random.seed(1234)
indices = list(range(len(df_iris)))
random.shuffle(indices)

split = int(0.8 * len(indices))
train_idx = indices[:split]
test_idx = indices[split:]

# Get raw train and test data
train_raw = df_iris.iloc[train_idx]
test_raw = df_iris.iloc[test_idx]

# Learn bins from training data
train_bins = {}
for column in features:
    train_bins[column] = learn_bins(train_raw[column].values)

# Learn bins from test data
test_bins = {}
for column in features:
    test_bins[column] = learn_bins(test_raw[column].values)

# Discretize train_data using train bins
train_data = []
for i in train_idx:
    row = []
    for column in features:
        value = df_iris.loc[i, column]
        discretized_value = discretize_value(value, train_bins[column])
        row.append(discretized_value)
    train_data.append(row)

# Discretize test_data using test bins
test_data = []
for i in test_idx:
    row = []
    for column in features:
        value = df_iris.loc[i, column]
        discretized_value = discretize_value(value, test_bins[column])
        row.append(discretized_value)
    test_data.append(row)

train_labels = [df_iris.loc[i, label] for i in train_idx]
test_labels = [df_iris.loc[i, label] for i in test_idx]

feature_indices = list(range(len(features)))
my_tree = id3(train_data, train_labels, feature_indices)

train_accuracy = accuracy(my_tree, train_data, train_labels)
test_accuracy = accuracy(my_tree, test_data, test_labels)

print(f'\nTrain accuracy: {train_accuracy:.1%}')
print(f'Test accuracy:  {test_accuracy:.1%}')


Train accuracy: 95.8%
Test accuracy:  90.0%


ID3 Decision Tree on the game dataset

In [10]:
df = pd.read_csv('popout.csv')

# Removing probability columns
df.drop(columns=["('drop', 0)", "('drop', 1)", "('drop', 2)", "('drop', 3)", "('drop', 4)", "('drop', 5)", "('drop', 6)", "('pop', 0)", 
                 "('pop', 1)", "('pop', 2)", "('pop', 3)", "('pop', 4)", "('pop', 5)", "('pop', 6)"], inplace=True)
df.columns
# New aggreagted dataset
state_cols = [c for c in df.columns if c != "best_action"]
agg = (df.groupby(state_cols, as_index=False)
         .agg({"best_action": lambda x: x.value_counts().idxmax()}))
agg.to_csv("popout_agg.csv", index=False)

/tmp/ipykernel_490362/2823401373.py:1: DtypeWarning: Columns (0: ('drop', 0), 1: ('drop', 1), 2: ('drop', 2), 3: ('drop', 3), 4: ('drop', 4), 5: ('drop', 5), 6: ('drop', 6), 7: ('pop', 0), 8: ('pop', 1), 9: ('pop', 2), 10: ('pop', 3), 11: ('pop', 4), 12: ('pop', 5), 13: ('pop', 6)) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('popout.csv')


In [11]:
df_new = pd.read_csv("popout_agg.csv")
label = "best_action"
state_cols = [c for c in df_new.columns if c != label]

# Aggregating: 2 top states for accuracy measure
top2_map = {}
for key, group in df.groupby(state_cols):
    top2 = [action for action, _ in Counter(group["best_action"].tolist()).most_common(1)]
    top2_map[key] = top2

def top2_accuracy(tree, data, state_cols_values, top2_map):
    correct = 0
    for i, row in enumerate(data):
        predicted = predict(tree, row)
        key = tuple(state_cols_values[i])  # must match the groupby key
        top2 = top2_map.get(key, [])
        if predicted in top2:
            correct += 1
    return correct / len(data) if data else 0.0

def str_to_action(s):
    if s is None:
        return None

    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        return None

def decode_state(row, col_names):
    """Rebuild a PopOut game from encoded feature row"""
    game = PopOut()
    for idx, val in enumerate(row):
        col = idx // 6
        r = idx % 6
        bit_index = col * 7 + r
        if val == "player":
            game.player |= (1 << bit_index)
        elif val == "opponent":
            game.opponent |= (1 << bit_index)
    return game

def valid_move_accuracy(tree, data):
    correct = 0
    for row in data:
        predicted = predict(tree, row)
        if predicted is None:
            continue
        predicted = str_to_action(predicted)
        game = decode_state(row, state_cols)
        if predicted in game.get_valid_actions():
            correct += 1
    return correct / len(data) if data else 0.0

random.seed(1234)
indices = list(range(len(df_new)))
random.shuffle(indices)

split = int(0.8 * len(indices))
train_idx = indices[:split]
test_idx = indices[split:]

train_labels = [df_new.loc[i, label] for i in train_idx]
test_labels = [df_new.loc[i, label] for i in test_idx]

train_data = df_new.loc[train_idx, state_cols].values.tolist()
test_data  = df_new.loc[test_idx,  state_cols].values.tolist()

feature_indices = list(range(len(state_cols)))
my_tree = id3(train_data, train_labels, feature_indices)
test_states = df_new.loc[test_idx, state_cols].values.tolist()
train_states = df_new.loc[train_idx, state_cols].values.tolist()

top2_acc = top2_accuracy(my_tree, train_data, train_states, top2_map)
moves_acc = valid_move_accuracy(my_tree, test_data)
print(f"Valid move accuracy: {moves_acc:.1%}")
print(f"Top-2 accuracy: {top2_acc:.1%}")

Valid move accuracy: 77.6%
Top-2 accuracy: 100.0%


In [19]:
import pickle
import sys
from src.players.id3_tree import DecisionTreeNode, id3

# Train on the full dataset (not just train split) for the final model - to play with MCTS
all_data   = df_new[state_cols].values.tolist()
all_labels = df_new[label].tolist()

sys.setrecursionlimit(10000)
full_tree = id3(all_data, all_labels, list(range(len(state_cols))))

with open('id3_model.pkl', 'wb') as f:
    pickle.dump((full_tree, state_cols), f)

print('Tree saved to id3_model.pkl')

Tree saved to id3_model.pkl
